## Simple Neural Network


In [1]:
import pandas as pd
from main import create_hog_colour_df, scale_data
from sklearn.model_selection import train_test_split
from constants import LEAKAGE_COLS
from sklearn.preprocessing import LabelEncoder
from main import one_hot_encode

colour_hog_df = create_hog_colour_df()
X = colour_hog_df.drop(columns=LEAKAGE_COLS)
Y = colour_hog_df["class_name"]

X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2718, stratify=Y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp, y_train_temp, test_size=0.2, random_state=2718, stratify=y_train_temp
)

X_train_scaled, X_val_scaled = scale_data(X_train=X_train, X_test=X_val)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
unique_classes = label_encoder.classes_

y_train_ohe = one_hot_encode(Y_ints=y_train_encoded, num_classes=len(unique_classes))

In [5]:
from nn_util import make_prediction, train_network, calculate_accuracy

# params_dict = train_network(
#     X=X_train_scaled,
#     hidden_size=64,
#     output_size=10,
#     epochs=1000,
#     Y=y_train_ohe,
#     learning_rate=1,
# )

learning_rates = [0.1, 0.2, 0.3, 0.4]
epoch_counts = [2500, 4000]
hidden_sizes = [32, 64, 128, 256]

best_hyperparams = {
    "best_lr": None,
    "best_ec": None,
    "best_hs": None,
    "best_accuracy": 0,
}

print("Starting search for best hyperparameters...")
total_num_tests = len(learning_rates) * len(epoch_counts) * len(hidden_sizes)
tests_completed = 0
for learning_rate in learning_rates:
    for epoch_count in epoch_counts:
        for hidden_size in hidden_sizes:
            proportion_completed = tests_completed / total_num_tests
            print(f"{100 * proportion_completed:.2f}% complete")
            params_dict = train_network(
                X=X_train_scaled,
                hidden_size=hidden_size,
                output_size=10,
                epochs=epoch_count,
                Y=y_train_ohe,
                learning_rate=learning_rate,
            )
            predictions, probs = make_prediction(
                X=X_val_scaled, classes_array=unique_classes, params_dict=params_dict
            )
            accuracy = calculate_accuracy(preds=predictions, true_labels=y_val)
            if accuracy > best_hyperparams["best_accuracy"]:
                best_hyperparams["best_accuracy"] = accuracy
                best_hyperparams["best_lr"] = learning_rate
                best_hyperparams["best_ec"] = epoch_count
                best_hyperparams["best_hs"] = hidden_size
                print(
                    f"New best! Accuracy: {accuracy * 100:.2f}% | LR: {learning_rate}, Epochs: {epoch_count}, HS: {hidden_size}"
                )
            tests_completed += 1


best_accuracy = best_hyperparams["best_accuracy"]
best_learning_rate = best_hyperparams["best_lr"]
best_epoch_count = best_hyperparams["best_ec"]
best_hidden_size = best_hyperparams["best_hs"]

print("Search over!")
print("Best hyperparameters are as follows:")

print(
    f"Accuracy: {best_accuracy * 100:.2f}% | LR: {best_learning_rate}, Epochs: {best_epoch_count}, HS: {best_hidden_size}"
)

# predictions, probs = make_prediction(
#     X=X_val_scaled, classes_array=unique_classes, params_dict=params_dict
# )

# df_of_preds = pd.DataFrame(predictions)
# df_of_probs = pd.DataFrame(probs)
# final_df = pd.concat(
#     [df_of_preds, df_of_probs], axis=1
# )  # adds it across row-wise, not just joined onto the bottom

# remember you have to test validation set on different learning rates and # of epochs
# accuracy = calculate_accuracy(preds=predictions, true_labels=y_val)
# print(f"Accuracy on validation set: {accuracy * 100:.2f}%")


# print(final_df)

Starting search for best hyperparameters...
0.00% complete
New best! Accuracy: 38.50% | LR: 0.1, Epochs: 2500, HS: 32
3.12% complete
New best! Accuracy: 42.50% | LR: 0.1, Epochs: 2500, HS: 64
6.25% complete
9.38% complete
New best! Accuracy: 44.33% | LR: 0.1, Epochs: 2500, HS: 256
12.50% complete
15.62% complete
18.75% complete
21.88% complete
25.00% complete
28.12% complete
31.25% complete
34.38% complete
37.50% complete
40.62% complete
43.75% complete
46.88% complete
50.00% complete
53.12% complete
56.25% complete
59.38% complete
62.50% complete
65.62% complete
68.75% complete
71.88% complete
75.00% complete
78.12% complete
81.25% complete
84.38% complete
87.50% complete
90.62% complete
93.75% complete
96.88% complete


KeyboardInterrupt: 